# NPC最终489例：B+C开发集 / A独立外部验证，生成并冻结4次重复五折固定划分

本Notebook读取最终v4 Frozen Master及既有v2预处理NPZ，但**不使用Master、dataset index或NPZ中的旧`cohort`定义**。

本次实验按`model_center`重新定义队列：**Center B + Center C = Development（309例）**，**Center A = External（180例）**。

Development内部按`model_center × severe_mucositis`联合分层生成`4 repeats × 5 folds`；每个outer train内部再生成80%/20%的分层inner train/validation。

Center A的180例始终完全锁定，不参与outer/inner划分、早停、epoch选择、模型选择、阈值或校准。

> NPZ影像数组不因队列重定义而改变。本版本继续复用原497例NPZ；NPZ中的旧`label`、`severe_mucositis`和旧`cohort`均不作为本次队列或标签来源。

> **BCdev_Aexternal_v2**：基于最新冻结Master重新生成固定4×5划分。Development=B+C=309（label 0/1=158/151；B=42/39，C=116/112），External=A=180（88/92）。输出目录采用锁定的BCdev_Aexternal_v2命名。

In [ ]:
# -*- coding: utf-8 -*-
r"""
NPC重度急性口腔黏膜炎3D dose-map项目
B+C开发 / A外部验证版：生成并冻结4次重复 × 5折交叉验证划分
===============================================================================

运行方式
--------
Jupyter Notebook：
    直接运行唯一代码单元。

Python脚本：
    python NPC_497_generate_fixed_repeated_5fold_4repeats_BCdev_Aexternal_v2.py

设计
----
本次实验队列（按model_center重新定义，不使用旧cohort字段）：
    Development：309例
        Center B：81例，label 0/1 = 42/39
        Center C：228例，label 0/1 = 116/112

    External：180例
        Center A：180例，label 0/1 = 88/92

Outer cross-validation：
    4个独立repeat
    每个repeat使用StratifiedKFold生成5个fold
    分层变量 = model_center × severe_mucositis

每个repeat中：
    309例Development患者恰好作为outer validation一次；
    其余4个fold中作为outer train；
    每例最终得到4次OOF预测。

Inner split：
    每个outer train内部再固定生成一次80%/20%的分层划分；
    输出inner_train.csv、inner_validation.csv；
    同时输出兼容旧训练代码的inner_earlystop.csv。

External：
    Center A的180例始终完全锁定；
    不参与outer/inner划分、早停、超参数选择、模型选择、阈值或校准。

输入
----
Master：
$NPC_PROJECT_ROOT\NPC_3DCNN_mucositis_master_frozen_BCdev_Aexternal_v2.xlsx

预处理数据：
$NPC_PROJECT_ROOT\
preprocessed_497_2x2x3_patch80x112x64_v2

输出
----
$NPC_PROJECT_ROOT\
fixed_splits_repeated_5fold_4repeats_497_BCdev_Aexternal_v2

每个fold目录：
repeat_01\fold_01
├─ outer_train.csv
├─ outer_validation.csv
├─ inner_train.csv
├─ inner_validation.csv
├─ inner_earlystop.csv
├─ external_validation.csv
├─ all_489_assignments.csv
├─ *_ids.txt
├─ fold_config.json
├─ fold_audit.txt
└─ fold_hashes_sha256.csv

根目录同时输出：
LABELS_LOCKED.csv
external_validation_locked_180.csv
all_20_folds_assignments_long.csv
outer_fold_assignments_wide.csv
outer_validation_frequency_4repeats.csv
fold_summary_20.csv
outer_stratum_summary_20.csv
inner_stratum_summary_20.csv
validation_overlap_counts_20x20.csv
validation_overlap_jaccard_20x20.csv
fixed_splits_4x5_manifest.xlsx
fixed_splits_4x5_config.json
file_hashes_sha256.csv
SPLITS_LOCKED.json
README_FIXED_SPLITS.txt
fixed_splits_4x5_run_log.txt

依赖
----
pip install numpy pandas scikit-learn openpyxl
"""

from pathlib import Path
import os
from datetime import datetime
import hashlib
import json
import shutil
import sys
import traceback

import numpy as np
import pandas as pd

try:
    from sklearn.model_selection import (
        StratifiedKFold,
        StratifiedShuffleSplit,
    )
except ImportError as error:
    raise ImportError(
        "缺少scikit-learn。请先运行：pip install scikit-learn"
    ) from error

try:
    from openpyxl import load_workbook
    from openpyxl.styles import (
        Alignment,
        Font,
        PatternFill,
    )
    from openpyxl.utils import get_column_letter
except ImportError as error:
    raise ImportError(
        "缺少openpyxl。请先运行：pip install openpyxl"
    ) from error


# =============================================================================
# 1. 路径与版本
# =============================================================================

def require_env_path(name: str) -> Path:
    """Return a required absolute path from an environment variable."""
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"Set {name} to an absolute path before running this notebook."
        )
    path = Path(value).expanduser()
    if not path.is_absolute():
        raise RuntimeError(f"{name} must be an absolute path: {value!r}")
    return path.resolve()

def optional_env_path(name: str, default: Path) -> Path:
    value = os.environ.get(name)
    if not value:
        return default.resolve()
    path = Path(value).expanduser()
    if not path.is_absolute():
        raise RuntimeError(f"{name} must be an absolute path: {value!r}")
    return path.resolve()


PROJECT_DIR = require_env_path("NPC_PROJECT_ROOT")

MASTER_XLSX = (
    PROJECT_DIR
    / "NPC_3DCNN_mucositis_master_frozen_BCdev_Aexternal_v2.xlsx"
)

PREPROCESSED_ROOT = (
    PROJECT_DIR
    / "preprocessed_497_2x2x3_patch80x112x64_v2"
)

DATASET_INDEX_CSV = (
    PREPROCESSED_ROOT
    / "dataset_index_497.csv"
)

NPZ_AUDIT_CSV = (
    PREPROCESSED_ROOT
    / "npz_audit_497.csv"
)

PREPROCESSING_CONFIG_JSON = (
    PREPROCESSED_ROOT
    / "preprocessing_config_497.json"
)

NPZ_DIR = (
    PREPROCESSED_ROOT
    / "npz"
)

OUTPUT_VERSION = (
    "BCdev_Aexternal_v2"
)

OUTPUT_ROOT = optional_env_path(
    "NPC_SPLIT_OUTPUT_ROOT",
    PROJECT_DIR
    / (
        "fixed_splits_repeated_5fold_"
        "4repeats_497_"
        + OUTPUT_VERSION
    ),
)

TEMP_ROOT = (
    OUTPUT_ROOT.parent
    / (
        OUTPUT_ROOT.name
        + "_INCOMPLETE"
    )
)

ALLOW_OVERWRITE_OUTPUT = False
AUDIT_NPZ_METADATA = True


# =============================================================================
# 2. Master和数据版本锁定
# =============================================================================

EXPECTED_MASTER_SHA256 = (
    "31a72c9d3e1b21d69b8b227eee5d21b9"
    "c2129624401f5d3176c0b1770e148c93"
)

# Excel文件重新保存可能改变字节级SHA256；本实验依靠病例/中心/标签内容审计锁定。
STRICT_MASTER_SHA256 = True

EXPECTED_NPZ_SHAPE_ZYX = (
    64,
    112,
    80,
)

REQUIRED_NPZ_CHANNELS = (
    "ct",
    "dose",
    "oral",
    "gtv",
)

REQUIRED_NPZ_METADATA_KEYS = (
    "patient_id",
    "model_center",
)

# 重要：
# 1. NPZ中的label和severe_mucositis属于旧预处理元数据，不读取、不比较；
# 2. NPZ中的cohort也是旧A+B开发/C外部定义，不作为本次队列来源；
# 3. 本次标签唯一来自MASTER_XLSX；
# 4. 本次cohort唯一由model_center映射：B/C -> Development，A -> External。


# =============================================================================
# 3. 固定划分参数
# =============================================================================

N_REPEATS = 4
N_OUTER_FOLDS = 5

OUTER_SPLIT_SEEDS = [
    20260701,
    20260702,
    20260703,
    20260704,
]

INNER_VALIDATION_FRACTION = 0.20

# inner seed = 50260711
# train seed = 30260711
def make_inner_split_seed(
    repeat_number: int,
    fold_number: int,
) -> int:
    return int(
        50260700
        + repeat_number * 10
        + fold_number
    )


def make_training_seed(
    repeat_number: int,
    fold_number: int,
) -> int:
    return int(
        30260700
        + repeat_number * 10
        + fold_number
    )


# =============================================================================
# 4. 严格预期
# =============================================================================

PREPROCESSING_EXCLUDED_ID = int(os.environ["NPC_PREPROCESSING_EXCLUDED_ID"])

EXPECTED_MASTER_TOTAL = 540
EXPECTED_MASTER_EXCLUDED = 51
EXPECTED_INCLUDED_TOTAL = 489
EXPECTED_PREPROCESSED_POOL_TOTAL = 497
EXPECTED_PREPROCESSED_EXTRA_TOTAL = 8

EXPECTED_COHORT_COUNTS = {
    "Development": 309,
    "External": 180,
}

EXPECTED_CENTER_COUNTS = {
    "A": 180,
    "B": 81,
    "C": 228,
}

EXPECTED_LABEL_COUNTS = {
    0: 246,
    1: 243,
}

EXPECTED_CENTER_LABEL_COUNTS = {
    ("A", 0): 88,
    ("A", 1): 92,
    ("B", 0): 42,
    ("B", 1): 39,
    ("C", 0): 116,
    ("C", 1): 112,
}

EXPECTED_DEVELOPMENT_STRATA = {
    "B_label0": 42,
    "B_label1": 39,
    "C_label0": 116,
    "C_label1": 112,
}

EXPECTED_EXTERNAL_LABEL_COUNTS = {
    0: 88,
    1: 92,
}

EXPECTED_TOTAL_FOLDS = (
    N_REPEATS
    * N_OUTER_FOLDS
)

EXPECTED_LONG_TABLE_ROWS = (
    EXPECTED_INCLUDED_TOTAL
    * EXPECTED_TOTAL_FOLDS
)

EXPECTED_DEV_OUTER_VALIDATION_FREQUENCY = (
    N_REPEATS
)

EXPECTED_DEV_OUTER_TRAIN_FREQUENCY = (
    N_REPEATS
    * (
        N_OUTER_FOLDS
        - 1
    )
)


# =============================================================================
# 5. 输出列
# =============================================================================

REQUIRED_SPLIT_COLUMNS = [
    "patient_id",
    "original_center",
    "model_center",
    "cohort",
    "severe_mucositis",
    "stratum",
    "npz_path",
]

OPTIONAL_BASELINE_COLUMNS = [
    "age",
    "sex(female=0,male=1)",
    "concurrent_chemotherapy（有=1，无=0）",
    "嗜酒史（有=1，无=0）",
    "吸烟史（有=1，无=0）",
    "height_cm",
    "weight_kg",
    "BMI",
    "T_stage",
    "N_stage",
    "M_stage",
    "clinical_stage",
    "prescription_dose",
    "fraction_dose",
    "fractions",
    "RT_technique",
    "RT_start_year",
]

# 明确不把治疗期间/治疗后的体重变化变量写入正式划分文件，
# 避免后续临床模型误用结局发生后的信息。
FORBIDDEN_POST_TREATMENT_COLUMNS = {
    "end_weight_kg",
    "weight_loss_kg",
    "weight_loss_percent",
}


# =============================================================================
# 6. 通用辅助函数
# =============================================================================

def is_blank(
    value,
) -> bool:
    if value is None:
        return True

    try:
        if pd.isna(
            value
        ):
            return True
    except Exception:
        pass

    return (
        str(
            value
        ).strip()
        == ""
    )


def patient_id_int(
    value,
) -> int:
    if is_blank(
        value
    ):
        raise ValueError(
            "发现空patient_id"
        )

    return int(
        float(
            str(
                value
            ).strip()
        )
    )


def normalize_center(
    value,
) -> str:
    center = (
        str(
            value
        )
        .strip()
        .upper()
    )

    if center not in {
        "A",
        "B",
        "C",
    }:
        raise ValueError(
            f"无效model_center：{value}"
        )

    return center


def normalize_cohort(
    value,
) -> str:
    cohort = (
        str(
            value
        )
        .strip()
        .lower()
        .replace(
            "_",
            " ",
        )
    )

    if cohort == "development":
        return "Development"

    if cohort in {
        "external",
        "external validation",
    }:
        return "External"

    raise ValueError(
        f"无效cohort：{value}"
    )


def normalize_label(
    value,
) -> int:
    label = int(
        float(
            value
        )
    )

    if label not in {
        0,
        1,
    }:
        raise ValueError(
            f"无效标签：{value}"
        )

    return label


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            block = file.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def scalar_text(
    value,
) -> str:
    array = np.asarray(
        value
    ).reshape(-1)

    if array.size != 1:
        return str(
            array
        )

    scalar = array[
        0
    ]

    if isinstance(
        scalar,
        bytes,
    ):
        return scalar.decode(
            "utf-8"
        )

    return str(
        scalar
    )


def scalar_int(
    value,
) -> int:
    array = np.asarray(
        value
    ).reshape(-1)

    if array.size != 1:
        raise ValueError(
            f"预期标量，但得到shape={np.asarray(value).shape}"
        )

    return int(
        array[
            0
        ]
    )


def dataframe_to_csv(
    dataframe: pd.DataFrame,
    path: Path,
) -> None:
    dataframe.to_csv(
        path,
        index=False,
        encoding="utf-8-sig",
    )


def write_json(
    payload,
    path: Path,
) -> None:
    path.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )


def write_id_txt(
    dataframe: pd.DataFrame,
    path: Path,
) -> None:
    path.write_text(
        "\n".join(
            str(
                int(
                    value
                )
            )
            for value
            in dataframe[
                "patient_id"
            ]
        ),
        encoding="utf-8",
    )


def style_excel(
    path: Path,
) -> None:
    workbook = load_workbook(
        path
    )

    header_fill = PatternFill(
        "solid",
        fgColor="1F4E78",
    )

    header_font = Font(
        bold=True,
        color="FFFFFF",
    )

    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = "A2"

        if (
            worksheet.max_row >= 1
            and
            worksheet.max_column >= 1
        ):
            worksheet.auto_filter.ref = (
                worksheet.dimensions
            )

        for cell in worksheet[
            1
        ]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

        for column_cells in worksheet.columns:
            values = [
                cell.value
                for cell
                in list(
                    column_cells
                )[
                    :1500
                ]
                if cell.value
                is not None
            ]

            max_length = max(
                (
                    len(
                        str(
                            value
                        )
                    )
                    for value
                    in values
                ),
                default=8,
            )

            worksheet.column_dimensions[
                get_column_letter(
                    column_cells[
                        0
                    ].column
                )
            ].width = min(
                max(
                    max_length + 2,
                    10,
                ),
                45,
            )

    workbook.save(
        path
    )


def count_by_stratum(
    dataframe: pd.DataFrame,
) -> dict:
    return (
        dataframe[
            "stratum"
        ]
        .value_counts()
        .sort_index()
        .astype(int)
        .to_dict()
    )


def check_stratification_balance(
    count_records,
    group_key: str,
    stratum_key: str,
    count_key: str,
    maximum_difference: int = 1,
) -> None:
    count_dataframe = pd.DataFrame(
        count_records
    )

    for stratum, group in count_dataframe.groupby(
        stratum_key
    ):
        counts = (
            group[
                count_key
            ]
            .astype(int)
            .tolist()
        )

        if (
            max(
                counts
            )
            - min(
                counts
            )
            > maximum_difference
        ):
            raise RuntimeError(
                f"{group_key}的{stratum}分层不平衡：{counts}"
            )


# =============================================================================
# 7. 输入和输出保护
# =============================================================================

required_paths = [
    MASTER_XLSX,
    PREPROCESSED_ROOT,
    DATASET_INDEX_CSV,
    NPZ_AUDIT_CSV,
    PREPROCESSING_CONFIG_JSON,
    NPZ_DIR,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            "找不到必要输入：\n"
            f"{required_path}"
        )

if len(
    OUTER_SPLIT_SEEDS
) != N_REPEATS:
    raise RuntimeError(
        "OUTER_SPLIT_SEEDS数量必须等于N_REPEATS"
    )

if len(
    set(
        OUTER_SPLIT_SEEDS
    )
) != N_REPEATS:
    raise RuntimeError(
        "OUTER_SPLIT_SEEDS必须互不相同"
    )

if OUTPUT_ROOT.exists():
    if not ALLOW_OVERWRITE_OUTPUT:
        raise FileExistsError(
            "正式输出目录已经存在：\n"
            f"{OUTPUT_ROOT}\n\n"
            "程序不会覆盖冻结结果。"
            "请确认旧目录后删除，"
            "或修改OUTPUT_VERSION。"
        )

    shutil.rmtree(
        OUTPUT_ROOT
    )

if TEMP_ROOT.exists():
    raise FileExistsError(
        "发现上次未完成的临时目录：\n"
        f"{TEMP_ROOT}\n\n"
        "请先检查并删除该目录后重新运行。"
    )


# =============================================================================
# 8. 读取并严格验证Frozen Master
# =============================================================================

master_sha256 = sha256_file(
    MASTER_XLSX
)

if (
    STRICT_MASTER_SHA256
    and
    master_sha256
    != EXPECTED_MASTER_SHA256
):
    raise RuntimeError(
        "Frozen Master SHA256不匹配，程序停止。\n\n"
        f"当前SHA256：{master_sha256}\n"
        f"预期SHA256：{EXPECTED_MASTER_SHA256}\n"
        f"文件：{MASTER_XLSX}"
    )

master = pd.read_excel(
    MASTER_XLSX,
    dtype=object,
    engine="openpyxl",
)

master.columns = [
    str(
        column
    ).strip()
    for column
    in master.columns
]

required_master_columns = {
    "patient_id",
    "original_center",
    "model_center",
    "exclude_reason",
    "severe_mucositis",
}

missing_master_columns = (
    required_master_columns
    - set(
        master.columns
    )
)

if missing_master_columns:
    raise KeyError(
        "Master缺少必要字段："
        f"{sorted(missing_master_columns)}"
    )

master[
    "patient_id_int"
] = master[
    "patient_id"
].map(
    patient_id_int
)

if master[
    "patient_id_int"
].duplicated().any():
    duplicate_ids = sorted(
        master.loc[
            master[
                "patient_id_int"
            ].duplicated(
                keep=False
            ),
            "patient_id_int",
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    raise RuntimeError(
        f"Master存在重复patient_id：{duplicate_ids}"
    )

included_mask = master[
    "exclude_reason"
].map(
    is_blank
)

included = master.loc[
    included_mask
].copy()

excluded = master.loc[
    ~included_mask
].copy()

included[
    "model_center_norm"
] = included[
    "model_center"
].map(
    normalize_center
)

# 本次实验不使用Master中的旧cohort字段。
# 唯一队列定义：B/C -> Development；A -> External。
included[
    "cohort_norm"
] = np.where(
    included[
        "model_center_norm"
    ].isin(
        [
            "B",
            "C",
        ]
    ),
    "Development",
    "External",
)

included[
    "label_int"
] = included[
    "severe_mucositis"
].map(
    normalize_label
)

master_checks = {
    "master_total": (
        len(
            master
        ),
        EXPECTED_MASTER_TOTAL,
    ),
    "master_excluded": (
        len(
            excluded
        ),
        EXPECTED_MASTER_EXCLUDED,
    ),
    "included_total": (
        len(
            included
        ),
        EXPECTED_INCLUDED_TOTAL,
    ),
}

for check_name, (
    actual,
    expected,
) in master_checks.items():
    if actual != expected:
        raise RuntimeError(
            f"{check_name}错误："
            f"实际={actual}，预期={expected}"
        )

if (
    PREPROCESSING_EXCLUDED_ID
    in set(
        included[
            "patient_id_int"
        ].astype(int)
    )
):
    raise RuntimeError(
        f"病例{PREPROCESSING_EXCLUDED_ID}"
        "不得进入最终489例"
    )

# cohort_norm为本次实验根据model_center重新生成的队列定义。
cohort_counts = (
    included[
        "cohort_norm"
    ]
    .value_counts()
    .sort_index()
    .astype(int)
    .to_dict()
)

center_counts = (
    included[
        "model_center_norm"
    ]
    .value_counts()
    .sort_index()
    .astype(int)
    .to_dict()
)

label_counts = (
    included[
        "label_int"
    ]
    .value_counts()
    .sort_index()
    .astype(int)
    .to_dict()
)

center_label_counts = (
    included.groupby(
        [
            "model_center_norm",
            "label_int",
        ]
    )
    .size()
    .astype(int)
    .to_dict()
)

if cohort_counts != EXPECTED_COHORT_COUNTS:
    raise RuntimeError(
        f"cohort数量错误：{cohort_counts}"
    )

if center_counts != EXPECTED_CENTER_COUNTS:
    raise RuntimeError(
        f"center数量错误：{center_counts}"
    )

if label_counts != EXPECTED_LABEL_COUNTS:
    raise RuntimeError(
        f"标签数量错误：{label_counts}"
    )

if (
    center_label_counts
    != EXPECTED_CENTER_LABEL_COUNTS
):
    raise RuntimeError(
        "中心×标签数量错误："
        f"{center_label_counts}"
    )

development = included.loc[
    included[
        "cohort_norm"
    ]
    == "Development"
].copy()

external = included.loc[
    included[
        "cohort_norm"
    ]
    == "External"
].copy()

if set(
    development[
        "model_center_norm"
    ].unique()
) != {
    "B",
    "C",
}:
    raise RuntimeError(
        "Development必须仅包含Center B和C"
    )

if set(
    external[
        "model_center_norm"
    ].unique()
) != {
    "A",
}:
    raise RuntimeError(
        "External必须仅包含Center A"
    )

development[
    "stratum"
] = (
    development[
        "model_center_norm"
    ]
    + "_label"
    + development[
        "label_int"
    ].astype(str)
)

external[
    "stratum"
] = (
    external[
        "model_center_norm"
    ]
    + "_label"
    + external[
        "label_int"
    ].astype(str)
)

development_strata = count_by_stratum(
    development
)

if (
    development_strata
    != EXPECTED_DEVELOPMENT_STRATA
):
    raise RuntimeError(
        "Development分层数量错误："
        f"{development_strata}"
    )


# =============================================================================
# 9. 验证预处理影像、dataset index及NPZ影像审计（忽略旧标签）
# =============================================================================

preprocessing_config = json.loads(
    PREPROCESSING_CONFIG_JSON.read_text(
        encoding="utf-8"
    )
)

config_master_sha256 = str(
    preprocessing_config.get(
        "master_sha256",
        "",
    )
).strip()

if (
    config_master_sha256
    and
    config_master_sha256
    != master_sha256
):
    print(
        "说明：预处理配置记录的是生成影像NPZ时的旧Master SHA256。"
    )
    print(
        "本次仅更新标签和分层划分，不重新生成影像数组；"
        "NPZ内旧标签不会被读取或比较。"
    )
    print(
        f"preprocessing_config_master_sha256={config_master_sha256}"
    )
    print(
        f"current_v4_master_sha256={master_sha256}"
    )

dataset_index = pd.read_csv(
    DATASET_INDEX_CSV,
    dtype=object,
    encoding="utf-8-sig",
)

index_pid_column = next(
    (
        column
        for column
        in [
            "patient_id_int",
            "patient_id",
            "id",
        ]
        if column
        in dataset_index.columns
    ),
    None,
)

if index_pid_column is None:
    raise KeyError(
        "dataset_index_497.csv中找不到patient_id字段"
    )

if (
    "output_npz_path"
    not in dataset_index.columns
):
    raise KeyError(
        "dataset_index_497.csv缺少output_npz_path"
    )

dataset_index[
    "_patient_id"
] = dataset_index[
    index_pid_column
].map(
    patient_id_int
)

if dataset_index[
    "_patient_id"
].duplicated().any():
    raise RuntimeError(
        "dataset_index存在重复patient_id"
    )

included_ids = set(
    included[
        "patient_id_int"
    ].astype(int)
)

index_ids = set(
    dataset_index[
        "_patient_id"
    ].astype(int)
)

if len(index_ids) != EXPECTED_PREPROCESSED_POOL_TOTAL:
    raise RuntimeError(
        f"预处理dataset index病例数错误：{len(index_ids)} != "
        f"{EXPECTED_PREPROCESSED_POOL_TOTAL}"
    )
if not included_ids.issubset(index_ids):
    raise RuntimeError(
        "最终489例未被预处理池完整覆盖：\n"
        f"缺少={sorted(included_ids - index_ids)}"
    )
preprocessed_extra_ids = index_ids - included_ids
excluded_ids = set(excluded["patient_id_int"].astype(int))
if (
    len(preprocessed_extra_ids) != EXPECTED_PREPROCESSED_EXTRA_TOTAL
    or not preprocessed_extra_ids.issubset(excluded_ids)
):
    raise RuntimeError(
        "预处理池与最终分析队列之间的8例差异不符合冻结Master：\n"
        f"extra={sorted(preprocessed_extra_ids)}"
    )
dataset_index_analysis = dataset_index.loc[
    dataset_index["_patient_id"].isin(included_ids)
].copy()

master_lookup = (
    included.set_index(
        "patient_id_int"
    )
)

# dataset_index仅核对patient_id和model_center。
# 其中旧标签与旧cohort均不读取、不比较。
# 本次cohort唯一由model_center重新定义。
index_optional_comparisons = {
    "model_center": (
        "model_center_norm",
        normalize_center,
    ),
}

for index_column, (
    master_column,
    normalizer,
) in index_optional_comparisons.items():
    if index_column not in dataset_index.columns:
        continue

    mismatches = []

    for _, row in dataset_index_analysis.iterrows():
        patient_id = int(
            row[
                "_patient_id"
            ]
        )

        actual = normalizer(
            row[
                index_column
            ]
        )

        expected = master_lookup.loc[
            patient_id,
            master_column,
        ]

        if actual != expected:
            mismatches.append(
                (
                    patient_id,
                    actual,
                    expected,
                )
            )

    if mismatches:
        raise RuntimeError(
            f"dataset index的{index_column}"
            f"与master不一致：{mismatches[:20]}"
        )

npz_path_map = {}

for _, row in dataset_index.iterrows():
    patient_id = int(
        row[
            "_patient_id"
        ]
    )

    raw_path = row[
        "output_npz_path"
    ]

    candidate_path = Path(
        str(
            raw_path
        ).strip()
    )

    if not candidate_path.exists():
        fallback_path = (
            NPZ_DIR
            / f"{patient_id}.npz"
        )

        if fallback_path.exists():
            candidate_path = fallback_path

    npz_path_map[
        patient_id
    ] = candidate_path

missing_npz = sorted(
    patient_id
    for patient_id, path
    in npz_path_map.items()
    if not path.exists()
)

if missing_npz:
    raise FileNotFoundError(
        "以下NPZ不存在："
        f"{missing_npz}"
    )

actual_npz_ids = {
    int(
        path.stem
    )
    for path
    in NPZ_DIR.glob(
        "*.npz"
    )
    if path.stem.isdigit()
}

if actual_npz_ids != index_ids:
    raise RuntimeError(
        "NPZ目录与497例预处理dataset index不一致：\n"
        f"缺少={sorted(index_ids - actual_npz_ids)}\n"
        f"多出={sorted(actual_npz_ids - index_ids)}"
    )
if not included_ids.issubset(actual_npz_ids):
    raise RuntimeError("最终489例未被NPZ池完整覆盖")

if (
    PREPROCESSING_EXCLUDED_ID
    in actual_npz_ids
):
    raise RuntimeError(
        "NPZ目录中出现配置的排除病例NPZ"
    )

existing_npz_audit = pd.read_csv(
    NPZ_AUDIT_CSV,
    dtype=object,
    encoding="utf-8-sig",
)

if (
    len(
        existing_npz_audit
    )
    != EXPECTED_PREPROCESSED_POOL_TOTAL
):
    raise RuntimeError(
        "既有NPZ审计病例数不是497"
    )

if (
    "audit_status"
    not in existing_npz_audit.columns
):
    raise KeyError(
        "npz_audit_497.csv缺少audit_status"
    )

existing_audit_failures = (
    existing_npz_audit.loc[
        existing_npz_audit[
            "audit_status"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        != "pass"
    ]
)

if len(
    existing_audit_failures
):
    raise RuntimeError(
        "既有NPZ审计存在失败病例"
    )


# =============================================================================
# 10. 独立复核最终489例对应的NPZ影像与病例元数据（忽略旧标签）
# =============================================================================

npz_metadata_audit_records = []

if AUDIT_NPZ_METADATA:
    print(
        "正在独立复核最终489例对应的NPZ的通道、shape、ID、中心和队列；NPZ旧标签不读取……"
    )

    for case_index, patient_id in enumerate(
        sorted(
            included_ids
        ),
        start=1,
    ):
        errors = []

        try:
            npz_path = npz_path_map[
                patient_id
            ]

            expected_row = master_lookup.loc[
                patient_id
            ]

            with np.load(
                npz_path,
                allow_pickle=False,
            ) as data:
                for key in (
                    REQUIRED_NPZ_CHANNELS
                    + REQUIRED_NPZ_METADATA_KEYS
                ):
                    if key not in data.files:
                        errors.append(
                            f"missing:{key}"
                        )

                if not errors:
                    for channel in REQUIRED_NPZ_CHANNELS:
                        array = np.asarray(
                            data[
                                channel
                            ]
                        )

                        if (
                            tuple(
                                array.shape
                            )
                            != EXPECTED_NPZ_SHAPE_ZYX
                        ):
                            errors.append(
                                f"{channel}_shape="
                                f"{tuple(array.shape)}"
                            )

                        if channel in {
                            "ct",
                            "dose",
                        }:
                            if not np.isfinite(
                                array
                            ).all():
                                errors.append(
                                    f"{channel}_nonfinite"
                                )

                        if channel in {
                            "oral",
                            "gtv",
                        }:
                            if int(
                                array.sum()
                            ) <= 0:
                                errors.append(
                                    f"{channel}_empty"
                                )

                    if (
                        scalar_int(
                            data[
                                "patient_id"
                            ]
                        )
                        != patient_id
                    ):
                        errors.append(
                            "patient_id_mismatch"
                        )

                    if (
                        scalar_text(
                            data[
                                "model_center"
                            ]
                        )
                        .strip()
                        .upper()
                        != expected_row[
                            "model_center_norm"
                        ]
                    ):
                        errors.append(
                            "model_center_mismatch"
                        )

                    # NPZ中的cohort、label、severe_mucositis均为旧元数据。
                    # 本次B+C开发/A外部实验有意不读取、不比较这些旧字段。
                    # patient_id、model_center及四个影像通道仍严格审计。

        except Exception:
            errors.append(
                traceback.format_exc()
            )

        npz_metadata_audit_records.append({
            "patient_id": (
                patient_id
            ),
            "npz_path": (
                str(
                    npz_path_map[
                        patient_id
                    ]
                )
            ),
            "status": (
                "PASS"
                if not errors
                else "FAIL"
            ),
            "errors": (
                " | ".join(
                    errors
                )
            ),
        })

        if (
            case_index % 50 == 0
            or
            case_index
            == EXPECTED_INCLUDED_TOTAL
        ):
            print(
                "NPZ复核进度："
                f"{case_index}/"
                f"{EXPECTED_INCLUDED_TOTAL}"
            )

    npz_metadata_audit_df = pd.DataFrame(
        npz_metadata_audit_records
    )

    npz_failures = (
        npz_metadata_audit_df.loc[
            npz_metadata_audit_df[
                "status"
            ]
            != "PASS"
        ]
    )

    if len(
        npz_failures
    ):
        raise RuntimeError(
            "NPZ内部复核失败：\n"
            + npz_failures.to_string(
                index=False
            )
        )

    print(
        "最终分析NPZ内部复核：489/489通过。"
    )

else:
    npz_metadata_audit_df = pd.DataFrame(
        columns=[
            "patient_id",
            "npz_path",
            "status",
            "errors",
        ]
    )


# =============================================================================
# 11. 生成基础病例表
# =============================================================================

available_optional_columns = [
    column
    for column
    in OPTIONAL_BASELINE_COLUMNS
    if column
    in included.columns
]

for forbidden_column in FORBIDDEN_POST_TREATMENT_COLUMNS:
    if (
        forbidden_column
        in available_optional_columns
    ):
        raise RuntimeError(
            f"禁止将治疗后变量写入划分：{forbidden_column}"
        )

base = included[
    [
        "patient_id_int",
        "original_center",
        "model_center_norm",
        "cohort_norm",
        "label_int",
    ]
    + available_optional_columns
].copy()

base = base.rename(
    columns={
        "patient_id_int": (
            "patient_id"
        ),
        "model_center_norm": (
            "model_center"
        ),
        "cohort_norm": (
            "cohort"
        ),
        "label_int": (
            "severe_mucositis"
        ),
    }
)

base[
    "original_center"
] = pd.to_numeric(
    base[
        "original_center"
    ],
    errors="raise",
).astype(int)

base[
    "patient_id"
] = pd.to_numeric(
    base[
        "patient_id"
    ],
    errors="raise",
).astype(int)

base[
    "severe_mucositis"
] = pd.to_numeric(
    base[
        "severe_mucositis"
    ],
    errors="raise",
).astype(int)

base[
    "stratum"
] = (
    base[
        "model_center"
    ]
    + "_label"
    + base[
        "severe_mucositis"
    ].astype(str)
)

base[
    "npz_path"
] = base[
    "patient_id"
].map(
    lambda patient_id:
    str(
        npz_path_map[
            int(
                patient_id
            )
        ]
    )
)

base = (
    base.sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)

# 把核心字段放在最前面。
ordered_columns = (
    REQUIRED_SPLIT_COLUMNS
    + [
        column
        for column
        in available_optional_columns
        if column
        not in REQUIRED_SPLIT_COLUMNS
    ]
)

base = base[
    ordered_columns
]

development_base = (
    base.loc[
        base[
            "cohort"
        ]
        == "Development"
    ]
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)

external_locked = (
    base.loc[
        base[
            "cohort"
        ]
        == "External"
    ]
    .sort_values(
        [
            "severe_mucositis",
            "patient_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

dev_ids = set(
    development_base[
        "patient_id"
    ].astype(int)
)

external_ids = set(
    external_locked[
        "patient_id"
    ].astype(int)
)

if (
    dev_ids
    & external_ids
):
    raise RuntimeError(
        "Development与External发生重叠"
    )

if (
    dev_ids
    | external_ids
) != included_ids:
    raise RuntimeError(
        "Development与External未完整覆盖489例"
    )

if (
    external_locked[
        "severe_mucositis"
    ]
    .value_counts()
    .sort_index()
    .astype(int)
    .to_dict()
    != EXPECTED_EXTERNAL_LABEL_COUNTS
):
    raise RuntimeError(
        "External标签分布错误"
    )


# =============================================================================
# 12. 创建临时输出目录和根级锁定文件
# =============================================================================

TEMP_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

labels_locked_path = (
    TEMP_ROOT
    / "LABELS_LOCKED.csv"
)

external_locked_path = (
    TEMP_ROOT
    / "external_validation_locked_180.csv"
)

dataframe_to_csv(
    base[
        REQUIRED_SPLIT_COLUMNS
    ],
    labels_locked_path,
)

dataframe_to_csv(
    external_locked,
    external_locked_path,
)

shutil.copy2(
    MASTER_XLSX,
    TEMP_ROOT
    / "master_frozen_used.xlsx",
)

shutil.copy2(
    DATASET_INDEX_CSV,
    TEMP_ROOT
    / "dataset_index_used.csv",
)

shutil.copy2(
    NPZ_AUDIT_CSV,
    TEMP_ROOT
    / "npz_audit_used.csv",
)

shutil.copy2(
    PREPROCESSING_CONFIG_JSON,
    TEMP_ROOT
    / "preprocessing_config_used.json",
)


# =============================================================================
# =============================================================================

long_assignment_tables = []
fold_summary_records = []
outer_stratum_records = []
inner_stratum_records = []

validation_sets = {}
outer_train_sets = {}
inner_validation_sets = {}

repeat_assignment_signatures = {}
repeat_fold_assignments = {}

fold_file_records = []

for repeat_number, outer_seed in enumerate(
    OUTER_SPLIT_SEEDS,
    start=1,
):
    repeat_name = (
        f"repeat_{repeat_number:02d}"
    )

    repeat_dir = (
        TEMP_ROOT
        / repeat_name
    )

    repeat_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    splitter = StratifiedKFold(
        n_splits=N_OUTER_FOLDS,
        shuffle=True,
        random_state=outer_seed,
    )

    repeat_validation_union = set()
    repeat_validation_sets = []
    repeat_fold_map = {}

    for fold_number, (
        outer_train_indices,
        outer_validation_indices,
    ) in enumerate(
        splitter.split(
            development_base,
            development_base[
                "stratum"
            ],
        ),
        start=1,
    ):
        fold_name = (
            f"fold_{fold_number:02d}"
        )

        split_name = (
            f"{repeat_name}_{fold_name}"
        )

        fold_dir = (
            repeat_dir
            / fold_name
        )

        fold_dir.mkdir(
            parents=True,
            exist_ok=False,
        )

        inner_seed = make_inner_split_seed(
            repeat_number,
            fold_number,
        )

        training_seed = make_training_seed(
            repeat_number,
            fold_number,
        )

        outer_train_df = (
            development_base.iloc[
                outer_train_indices
            ]
            .copy()
            .sort_values(
                [
                    "model_center",
                    "severe_mucositis",
                    "patient_id",
                ]
            )
            .reset_index(
                drop=True
            )
        )

        outer_validation_df = (
            development_base.iloc[
                outer_validation_indices
            ]
            .copy()
            .sort_values(
                [
                    "model_center",
                    "severe_mucositis",
                    "patient_id",
                ]
            )
            .reset_index(
                drop=True
            )
        )

        outer_train_ids = set(
            outer_train_df[
                "patient_id"
            ].astype(int)
        )

        outer_validation_ids = set(
            outer_validation_df[
                "patient_id"
            ].astype(int)
        )

        if (
            outer_train_ids
            & outer_validation_ids
        ):
            raise RuntimeError(
                f"{split_name}的outer train/validation重叠"
            )

        if (
            outer_train_ids
            | outer_validation_ids
        ) != dev_ids:
            raise RuntimeError(
                f"{split_name}未完整覆盖Development"
            )

        if (
            outer_train_ids
            | outer_validation_ids
        ) & external_ids:
            raise RuntimeError(
                f"{split_name}混入External病例"
            )

        if (
            PREPROCESSING_EXCLUDED_ID
            in (
                outer_train_ids
                | outer_validation_ids
                | external_ids
            )
        ):
            raise RuntimeError(
                f"{split_name}出现配置的排除病例"
            )

        if (
            repeat_validation_union
            & outer_validation_ids
        ):
            raise RuntimeError(
                f"{repeat_name}内不同fold的"
                "outer validation发生重叠"
            )

        repeat_validation_union.update(
            outer_validation_ids
        )

        repeat_validation_sets.append(
            outer_validation_ids
        )

        for patient_id in outer_validation_ids:
            repeat_fold_map[
                patient_id
            ] = fold_number

        # -----------------------------------------------------
        # 固定inner split
        # -----------------------------------------------------

        inner_splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=(
                INNER_VALIDATION_FRACTION
            ),
            random_state=inner_seed,
        )

        (
            inner_train_indices,
            inner_validation_indices,
        ) = next(
            inner_splitter.split(
                outer_train_df,
                outer_train_df[
                    "stratum"
                ],
            )
        )

        inner_train_df = (
            outer_train_df.iloc[
                inner_train_indices
            ]
            .copy()
            .sort_values(
                [
                    "model_center",
                    "severe_mucositis",
                    "patient_id",
                ]
            )
            .reset_index(
                drop=True
            )
        )

        inner_validation_df = (
            outer_train_df.iloc[
                inner_validation_indices
            ]
            .copy()
            .sort_values(
                [
                    "model_center",
                    "severe_mucositis",
                    "patient_id",
                ]
            )
            .reset_index(
                drop=True
            )
        )

        inner_train_ids = set(
            inner_train_df[
                "patient_id"
            ].astype(int)
        )

        inner_validation_ids = set(
            inner_validation_df[
                "patient_id"
            ].astype(int)
        )

        if (
            inner_train_ids
            & inner_validation_ids
        ):
            raise RuntimeError(
                f"{split_name}的inner train/validation重叠"
            )

        if (
            inner_train_ids
            | inner_validation_ids
        ) != outer_train_ids:
            raise RuntimeError(
                f"{split_name}的inner split"
                "未完整覆盖outer train"
            )

        if (
            inner_train_ids
            | inner_validation_ids
        ) & outer_validation_ids:
            raise RuntimeError(
                f"{split_name}的inner split"
                "混入outer validation"
            )

        if (
            inner_train_ids
            | inner_validation_ids
        ) & external_ids:
            raise RuntimeError(
                f"{split_name}的inner split"
                "混入External"
            )

        # 每个inner集合必须包含全部4个Development strata。
        if set(
            inner_train_df[
                "stratum"
            ].unique()
        ) != set(
            EXPECTED_DEVELOPMENT_STRATA
        ):
            raise RuntimeError(
                f"{split_name}的inner train"
                "未包含全部4个strata"
            )

        if set(
            inner_validation_df[
                "stratum"
            ].unique()
        ) != set(
            EXPECTED_DEVELOPMENT_STRATA
        ):
            raise RuntimeError(
                f"{split_name}的inner validation"
                "未包含全部4个strata"
            )

        # -----------------------------------------------------
        # -----------------------------------------------------

        fold_metadata = {
            "repeat_number": (
                repeat_number
            ),
            "fold_number": (
                fold_number
            ),
            "split_name": (
                split_name
            ),
            "outer_split_seed": (
                outer_seed
            ),
            "inner_split_seed": (
                inner_seed
            ),
            "training_seed": (
                training_seed
            ),
        }

        def add_fold_metadata(
            dataframe: pd.DataFrame,
            split_role: str,
        ) -> pd.DataFrame:
            output = dataframe.copy()

            for key, value in (
                fold_metadata.items()
            ):
                output[
                    key
                ] = value

            output[
                "split_role"
            ] = split_role

            return output

        outer_train_output = add_fold_metadata(
            outer_train_df,
            "outer_train",
        )

        outer_validation_output = (
            add_fold_metadata(
                outer_validation_df,
                "outer_validation",
            )
        )

        inner_train_output = add_fold_metadata(
            inner_train_df,
            "inner_train",
        )

        inner_validation_output = (
            add_fold_metadata(
                inner_validation_df,
                "inner_validation",
            )
        )

        # -----------------------------------------------------
        # -----------------------------------------------------

        outer_train_path = (
            fold_dir
            / "outer_train.csv"
        )

        outer_validation_path = (
            fold_dir
            / "outer_validation.csv"
        )

        inner_train_path = (
            fold_dir
            / "inner_train.csv"
        )

        inner_validation_path = (
            fold_dir
            / "inner_validation.csv"
        )

        inner_earlystop_path = (
            fold_dir
            / "inner_earlystop.csv"
        )

        external_path = (
            fold_dir
            / "external_validation.csv"
        )

        all_assignment_path = (
            fold_dir
            / "all_489_assignments.csv"
        )

        dataframe_to_csv(
            outer_train_output,
            outer_train_path,
        )

        dataframe_to_csv(
            outer_validation_output,
            outer_validation_path,
        )

        dataframe_to_csv(
            inner_train_output,
            inner_train_path,
        )

        dataframe_to_csv(
            inner_validation_output,
            inner_validation_path,
        )

        # 兼容旧工程代码，文件内容与inner_validation.csv完全相同。
        shutil.copy2(
            inner_validation_path,
            inner_earlystop_path,
        )

        shutil.copy2(
            external_locked_path,
            external_path,
        )

        write_id_txt(
            outer_train_df,
            fold_dir
            / "outer_train_ids.txt",
        )

        write_id_txt(
            outer_validation_df,
            fold_dir
            / "outer_validation_ids.txt",
        )

        write_id_txt(
            inner_train_df,
            fold_dir
            / "inner_train_ids.txt",
        )

        write_id_txt(
            inner_validation_df,
            fold_dir
            / "inner_validation_ids.txt",
        )

        shutil.copy2(
            fold_dir
            / "inner_validation_ids.txt",
            fold_dir
            / "inner_earlystop_ids.txt",
        )

        write_id_txt(
            external_locked,
            fold_dir
            / "external_validation_ids.txt",
        )

        # -----------------------------------------------------
        # 生成489例最终分析角色总表
        # -----------------------------------------------------

        assignment = base.copy()

        assignment[
            "repeat_number"
        ] = repeat_number

        assignment[
            "fold_number"
        ] = fold_number

        assignment[
            "split_name"
        ] = split_name

        assignment[
            "outer_split_seed"
        ] = outer_seed

        assignment[
            "inner_split_seed"
        ] = inner_seed

        assignment[
            "training_seed"
        ] = training_seed

        def determine_outer_role(
            patient_id,
        ) -> str:
            patient_id = int(
                patient_id
            )

            if patient_id in outer_train_ids:
                return "outer_train"

            if patient_id in outer_validation_ids:
                return "outer_validation"

            if patient_id in external_ids:
                return "external"

            raise RuntimeError(
                f"{split_name}中的病例"
                f"{patient_id}未分配outer role"
            )

        def determine_inner_role(
            patient_id,
        ) -> str:
            patient_id = int(
                patient_id
            )

            if patient_id in inner_train_ids:
                return "inner_train"

            if patient_id in inner_validation_ids:
                return "inner_validation"

            if patient_id in outer_validation_ids:
                return "not_applicable_outer_validation"

            if patient_id in external_ids:
                return "not_applicable_external"

            raise RuntimeError(
                f"{split_name}中的病例"
                f"{patient_id}未分配inner role"
            )

        assignment[
            "outer_role"
        ] = assignment[
            "patient_id"
        ].map(
            determine_outer_role
        )

        assignment[
            "inner_role"
        ] = assignment[
            "patient_id"
        ].map(
            determine_inner_role
        )

        assignment[
            "is_outer_train"
        ] = (
            assignment[
                "outer_role"
            ]
            == "outer_train"
        ).astype(int)

        assignment[
            "is_outer_validation"
        ] = (
            assignment[
                "outer_role"
            ]
            == "outer_validation"
        ).astype(int)

        assignment[
            "is_inner_train"
        ] = (
            assignment[
                "inner_role"
            ]
            == "inner_train"
        ).astype(int)

        assignment[
            "is_inner_validation"
        ] = (
            assignment[
                "inner_role"
            ]
            == "inner_validation"
        ).astype(int)

        assignment[
            "is_external"
        ] = (
            assignment[
                "outer_role"
            ]
            == "external"
        ).astype(int)

        dataframe_to_csv(
            assignment,
            all_assignment_path,
        )

        # -----------------------------------------------------
        # -----------------------------------------------------

        outer_train_strata = (
            count_by_stratum(
                outer_train_df
            )
        )

        outer_validation_strata = (
            count_by_stratum(
                outer_validation_df
            )
        )

        inner_train_strata = (
            count_by_stratum(
                inner_train_df
            )
        )

        inner_validation_strata = (
            count_by_stratum(
                inner_validation_df
            )
        )

        for stratum in sorted(
            EXPECTED_DEVELOPMENT_STRATA
        ):
            outer_stratum_records.append({
                "repeat_number": (
                    repeat_number
                ),
                "fold_number": (
                    fold_number
                ),
                "split_name": (
                    split_name
                ),
                "stratum": (
                    stratum
                ),
                "outer_train_n": (
                    int(
                        outer_train_strata.get(
                            stratum,
                            0,
                        )
                    )
                ),
                "outer_validation_n": (
                    int(
                        outer_validation_strata.get(
                            stratum,
                            0,
                        )
                    )
                ),
            })

            inner_stratum_records.append({
                "repeat_number": (
                    repeat_number
                ),
                "fold_number": (
                    fold_number
                ),
                "split_name": (
                    split_name
                ),
                "stratum": (
                    stratum
                ),
                "inner_train_n": (
                    int(
                        inner_train_strata.get(
                            stratum,
                            0,
                        )
                    )
                ),
                "inner_validation_n": (
                    int(
                        inner_validation_strata.get(
                            stratum,
                            0,
                        )
                    )
                ),
            })

        fold_summary_records.append({
            "repeat_number": (
                repeat_number
            ),
            "fold_number": (
                fold_number
            ),
            "split_name": (
                split_name
            ),
            "outer_split_seed": (
                outer_seed
            ),
            "inner_split_seed": (
                inner_seed
            ),
            "training_seed": (
                training_seed
            ),
            "outer_train_n": (
                len(
                    outer_train_df
                )
            ),
            "outer_validation_n": (
                len(
                    outer_validation_df
                )
            ),
            "inner_train_n": (
                len(
                    inner_train_df
                )
            ),
            "inner_validation_n": (
                len(
                    inner_validation_df
                )
            ),
            "external_n": (
                len(
                    external_locked
                )
            ),
            "outer_train_label0": (
                int(
                    (
                        outer_train_df[
                            "severe_mucositis"
                        ]
                        == 0
                    ).sum()
                )
            ),
            "outer_train_label1": (
                int(
                    (
                        outer_train_df[
                            "severe_mucositis"
                        ]
                        == 1
                    ).sum()
                )
            ),
            "outer_validation_label0": (
                int(
                    (
                        outer_validation_df[
                            "severe_mucositis"
                        ]
                        == 0
                    ).sum()
                )
            ),
            "outer_validation_label1": (
                int(
                    (
                        outer_validation_df[
                            "severe_mucositis"
                        ]
                        == 1
                    ).sum()
                )
            ),
            "status": (
                "PASS"
            ),
        })

        validation_sets[
            split_name
        ] = outer_validation_ids

        outer_train_sets[
            split_name
        ] = outer_train_ids

        inner_validation_sets[
            split_name
        ] = inner_validation_ids

        long_assignment_tables.append(
            assignment
        )

        fold_config = {
            "created_at": (
                datetime.now().isoformat(
                    timespec="seconds"
                )
            ),
            "repeat_number": (
                repeat_number
            ),
            "fold_number": (
                fold_number
            ),
            "split_name": (
                split_name
            ),
            "outer_split_method": (
                "StratifiedKFold"
            ),
            "outer_n_splits": (
                N_OUTER_FOLDS
            ),
            "outer_shuffle": (
                True
            ),
            "outer_split_seed": (
                outer_seed
            ),
            "inner_split_method": (
                "StratifiedShuffleSplit"
            ),
            "inner_validation_fraction": (
                INNER_VALIDATION_FRACTION
            ),
            "inner_split_seed": (
                inner_seed
            ),
            "training_seed": (
                training_seed
            ),
            "stratification": (
                "model_center × severe_mucositis"
            ),
            "outer_train_n": (
                len(
                    outer_train_df
                )
            ),
            "outer_validation_n": (
                len(
                    outer_validation_df
                )
            ),
            "inner_train_n": (
                len(
                    inner_train_df
                )
            ),
            "inner_validation_n": (
                len(
                    inner_validation_df
                )
            ),
            "external_n": (
                len(
                    external_locked
                )
            ),
            "external_center": (
                "A"
            ),
            "external_locked": (
                True
            ),
            "master_sha256": (
                master_sha256
            ),
            "labels_locked_sha256": (
                sha256_file(
                    labels_locked_path
                )
            ),
            "configured_excluded_case_present": (
                False
            ),
        }

        fold_config_path = (
            fold_dir
            / "fold_config.json"
        )

        write_json(
            fold_config,
            fold_config_path,
        )

        fold_audit_lines = [
            (
                f"Split: {split_name}"
            ),
            (
                "=" * 80
            ),
            (
                f"Outer split seed: {outer_seed}"
            ),
            (
                f"Inner split seed: {inner_seed}"
            ),
            (
                f"Training seed: {training_seed}"
            ),
            (
                f"Outer train: {len(outer_train_df)}"
            ),
            (
                "Outer validation: "
                f"{len(outer_validation_df)}"
            ),
            (
                f"Inner train: {len(inner_train_df)}"
            ),
            (
                "Inner validation: "
                f"{len(inner_validation_df)}"
            ),
            (
                f"External: {len(external_locked)}"
            ),
            (
                f"Outer train strata: {outer_train_strata}"
            ),
            (
                "Outer validation strata: "
                f"{outer_validation_strata}"
            ),
            (
                f"Inner train strata: {inner_train_strata}"
            ),
            (
                "Inner validation strata: "
                f"{inner_validation_strata}"
            ),
            (
                "Development complete: True"
            ),
            (
                "External contamination: False"
            ),
            (
                "Configured excluded case present: False"
            ),
            (
                "Status: PASS"
            ),
        ]

        (
            fold_dir
            / "fold_audit.txt"
        ).write_text(
            "\n".join(
                fold_audit_lines
            ),
            encoding="utf-8",
        )

        fold_files_before_hash = sorted(
            path
            for path
            in fold_dir.iterdir()
            if (
                path.is_file()
                and
                path.name
                != "fold_hashes_sha256.csv"
            )
        )

        fold_hash_dataframe = pd.DataFrame([
            {
                "file_name": (
                    path.name
                ),
                "size_bytes": (
                    path.stat().st_size
                ),
                "sha256": (
                    sha256_file(
                        path
                    )
                ),
            }
            for path
            in fold_files_before_hash
        ])

        fold_hash_path = (
            fold_dir
            / "fold_hashes_sha256.csv"
        )

        dataframe_to_csv(
            fold_hash_dataframe,
            fold_hash_path,
        )

        fold_file_records.append({
            "split_name": (
                split_name
            ),
            "fold_dir": (
                str(
                    fold_dir
                )
            ),
            "outer_train_sha256": (
                sha256_file(
                    outer_train_path
                )
            ),
            "outer_validation_sha256": (
                sha256_file(
                    outer_validation_path
                )
            ),
            "inner_train_sha256": (
                sha256_file(
                    inner_train_path
                )
            ),
            "inner_validation_sha256": (
                sha256_file(
                    inner_validation_path
                )
            ),
            "inner_earlystop_sha256": (
                sha256_file(
                    inner_earlystop_path
                )
            ),
            "external_validation_sha256": (
                sha256_file(
                    external_path
                )
            ),
            "all_assignments_sha256": (
                sha256_file(
                    all_assignment_path
                )
            ),
            "fold_config_sha256": (
                sha256_file(
                    fold_config_path
                )
            ),
            "fold_hash_manifest_sha256": (
                sha256_file(
                    fold_hash_path
                )
            ),
        })

    if (
        repeat_validation_union
        != dev_ids
    ):
        raise RuntimeError(
            f"{repeat_name}的5个outer validation"
            "未恰好覆盖全部309例Development"
        )

    if len(
        repeat_validation_sets
    ) != N_OUTER_FOLDS:
        raise RuntimeError(
            f"{repeat_name}的fold数量错误"
        )

    repeat_assignment_text = "|".join(
        f"{patient_id}:{repeat_fold_map[patient_id]}"
        for patient_id
        in sorted(
            repeat_fold_map
        )
    )

    repeat_assignment_signatures[
        repeat_name
    ] = hashlib.sha256(
        repeat_assignment_text.encode(
            "utf-8"
        )
    ).hexdigest()

    repeat_fold_assignments[
        repeat_name
    ] = repeat_fold_map

if len(
    set(
        repeat_assignment_signatures.values()
    )
) != N_REPEATS:
    raise RuntimeError(
        "4个repeat的完整fold assignment并非全部不同"
    )

if len(
    validation_sets
) != EXPECTED_TOTAL_FOLDS:
    raise RuntimeError(
        "生成的outer fold总数不是20"
    )

validation_signatures = {
    split_name: hashlib.sha256(
        ",".join(
            str(
                patient_id
            )
            for patient_id
            in sorted(
                patient_ids
            )
        ).encode(
            "utf-8"
        )
    ).hexdigest()
    for split_name, patient_ids
    in validation_sets.items()
}

if len(
    set(
        validation_signatures.values()
    )
) != EXPECTED_TOTAL_FOLDS:
    raise RuntimeError(
        "20个outer validation集合并非全部不同"
    )


# =============================================================================
# 14. 全局汇总与交叉审计
# =============================================================================

all_assignments_long = (
    pd.concat(
        long_assignment_tables,
        ignore_index=True,
    )
    .sort_values(
        [
            "repeat_number",
            "fold_number",
            "patient_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

if len(
    all_assignments_long
) != EXPECTED_LONG_TABLE_ROWS:
    raise RuntimeError(
        "长表行数错误："
        f"{len(all_assignments_long)}，"
        f"预期{EXPECTED_LONG_TABLE_ROWS}"
    )

fold_summary_df = (
    pd.DataFrame(
        fold_summary_records
    )
    .sort_values(
        [
            "repeat_number",
            "fold_number",
        ]
    )
    .reset_index(
        drop=True
    )
)

outer_stratum_df = (
    pd.DataFrame(
        outer_stratum_records
    )
    .sort_values(
        [
            "repeat_number",
            "fold_number",
            "stratum",
        ]
    )
    .reset_index(
        drop=True
    )
)

inner_stratum_df = (
    pd.DataFrame(
        inner_stratum_records
    )
    .sort_values(
        [
            "repeat_number",
            "fold_number",
            "stratum",
        ]
    )
    .reset_index(
        drop=True
    )
)

check_stratification_balance(
    outer_stratum_records,
    group_key="outer validation",
    stratum_key="stratum",
    count_key="outer_validation_n",
    maximum_difference=1,
)

for repeat_number in range(
    1,
    N_REPEATS + 1,
):
    repeat_outer_strata = (
        outer_stratum_df.loc[
            outer_stratum_df[
                "repeat_number"
            ]
            == repeat_number
        ]
    )

    accumulated = (
        repeat_outer_strata.groupby(
            "stratum"
        )[
            "outer_validation_n"
        ]
        .sum()
        .astype(int)
        .to_dict()
    )

    if (
        accumulated
        != EXPECTED_DEVELOPMENT_STRATA
    ):
        raise RuntimeError(
            f"repeat_{repeat_number:02d}"
            f"的分层累计错误：{accumulated}"
        )

# ------------------------------------------------------------
# Outer validation频率
# ------------------------------------------------------------

frequency_records = []

for _, row in base.iterrows():
    patient_id = int(
        row[
            "patient_id"
        ]
    )

    outer_validation_count = sum(
        patient_id
        in patient_ids
        for patient_ids
        in validation_sets.values()
    )

    outer_train_count = sum(
        patient_id
        in patient_ids
        for patient_ids
        in outer_train_sets.values()
    )

    inner_validation_count = sum(
        patient_id
        in patient_ids
        for patient_ids
        in inner_validation_sets.values()
    )

    expected_outer_validation = (
        EXPECTED_DEV_OUTER_VALIDATION_FREQUENCY
        if row[
            "cohort"
        ]
        == "Development"
        else 0
    )

    expected_outer_train = (
        EXPECTED_DEV_OUTER_TRAIN_FREQUENCY
        if row[
            "cohort"
        ]
        == "Development"
        else 0
    )

    frequency_records.append({
        "patient_id": (
            patient_id
        ),
        "model_center": (
            row[
                "model_center"
            ]
        ),
        "cohort": (
            row[
                "cohort"
            ]
        ),
        "severe_mucositis": (
            int(
                row[
                    "severe_mucositis"
                ]
            )
        ),
        "stratum": (
            row[
                "stratum"
            ]
        ),
        "outer_validation_count_across_20": (
            outer_validation_count
        ),
        "expected_outer_validation_count": (
            expected_outer_validation
        ),
        "outer_train_count_across_20": (
            outer_train_count
        ),
        "expected_outer_train_count": (
            expected_outer_train
        ),
        "inner_validation_count_across_20": (
            inner_validation_count
        ),
        "status": (
            "PASS"
            if (
                outer_validation_count
                == expected_outer_validation
                and
                outer_train_count
                == expected_outer_train
            )
            else "FAIL"
        ),
    })

frequency_df = (
    pd.DataFrame(
        frequency_records
    )
    .sort_values(
        [
            "cohort",
            "model_center",
            "severe_mucositis",
            "patient_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

frequency_failures = (
    frequency_df.loc[
        frequency_df[
            "status"
        ]
        != "PASS"
    ]
)

if len(
    frequency_failures
):
    raise RuntimeError(
        "部分病例的outer validation/train频率错误：\n"
        + frequency_failures.to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# ------------------------------------------------------------

wide_assignments = base.copy()

for repeat_number in range(
    1,
    N_REPEATS + 1,
):
    repeat_name = (
        f"repeat_{repeat_number:02d}"
    )

    fold_map = repeat_fold_assignments[
        repeat_name
    ]

    wide_assignments[
        f"{repeat_name}_outer_fold"
    ] = wide_assignments[
        "patient_id"
    ].map(
        lambda patient_id:
        (
            int(
                fold_map[
                    int(
                        patient_id
                    )
                ]
            )
            if int(
                patient_id
            )
            in fold_map
            else "External"
        )
    )

# ------------------------------------------------------------
# 20个outer validation集合重叠矩阵
# ------------------------------------------------------------

split_names = sorted(
    validation_sets
)

overlap_counts = pd.DataFrame(
    index=split_names,
    columns=split_names,
    dtype=int,
)

overlap_jaccard = pd.DataFrame(
    index=split_names,
    columns=split_names,
    dtype=float,
)

for first_name in split_names:
    for second_name in split_names:
        first_set = validation_sets[
            first_name
        ]

        second_set = validation_sets[
            second_name
        ]

        intersection_size = len(
            first_set
            & second_set
        )

        union_size = len(
            first_set
            | second_set
        )

        overlap_counts.loc[
            first_name,
            second_name,
        ] = intersection_size

        overlap_jaccard.loc[
            first_name,
            second_name,
        ] = (
            intersection_size
            / union_size
        )

overlap_counts = (
    overlap_counts.astype(int)
)

for repeat_number in range(
    1,
    N_REPEATS + 1,
):
    current_names = [
        f"repeat_{repeat_number:02d}_fold_{fold_number:02d}"
        for fold_number
        in range(
            1,
            N_OUTER_FOLDS + 1,
        )
    ]

    for first_index, first_name in enumerate(
        current_names
    ):
        for second_name in current_names[
            first_index + 1:
        ]:
            if int(
                overlap_counts.loc[
                    first_name,
                    second_name,
                ]
            ) != 0:
                raise RuntimeError(
                    f"{first_name}和{second_name}"
                    "在同一repeat内发生重叠"
                )


# =============================================================================
# 15. 保存根级汇总文件
# =============================================================================

all_assignments_long_path = (
    TEMP_ROOT
    / "all_20_folds_assignments_long.csv"
)

wide_assignments_path = (
    TEMP_ROOT
    / "outer_fold_assignments_wide.csv"
)

frequency_path = (
    TEMP_ROOT
    / "outer_validation_frequency_4repeats.csv"
)

fold_summary_path = (
    TEMP_ROOT
    / "fold_summary_20.csv"
)

outer_stratum_path = (
    TEMP_ROOT
    / "outer_stratum_summary_20.csv"
)

inner_stratum_path = (
    TEMP_ROOT
    / "inner_stratum_summary_20.csv"
)

overlap_counts_path = (
    TEMP_ROOT
    / "validation_overlap_counts_20x20.csv"
)

overlap_jaccard_path = (
    TEMP_ROOT
    / "validation_overlap_jaccard_20x20.csv"
)

fold_files_manifest_path = (
    TEMP_ROOT
    / "fold_file_hashes_summary.csv"
)

dataframe_to_csv(
    all_assignments_long,
    all_assignments_long_path,
)

dataframe_to_csv(
    wide_assignments,
    wide_assignments_path,
)

dataframe_to_csv(
    frequency_df,
    frequency_path,
)

dataframe_to_csv(
    fold_summary_df,
    fold_summary_path,
)

dataframe_to_csv(
    outer_stratum_df,
    outer_stratum_path,
)

dataframe_to_csv(
    inner_stratum_df,
    inner_stratum_path,
)

overlap_counts.to_csv(
    overlap_counts_path,
    encoding="utf-8-sig",
)

overlap_jaccard.to_csv(
    overlap_jaccard_path,
    encoding="utf-8-sig",
)

fold_files_manifest_df = (
    pd.DataFrame(
        fold_file_records
    )
    .sort_values(
        "split_name"
    )
    .reset_index(
        drop=True
    )
)

dataframe_to_csv(
    fold_files_manifest_df,
    fold_files_manifest_path,
)


# =============================================================================
# 16. 最终审计
# =============================================================================

final_audit_records = [
    {
        "check": (
            "master_total"
        ),
        "actual": (
            len(
                master
            )
        ),
        "expected": (
            EXPECTED_MASTER_TOTAL
        ),
    },
    {
        "check": (
            "master_excluded"
        ),
        "actual": (
            len(
                excluded
            )
        ),
        "expected": (
            EXPECTED_MASTER_EXCLUDED
        ),
    },
    {
        "check": (
            "included_total"
        ),
        "actual": (
            len(
                included
            )
        ),
        "expected": (
            EXPECTED_INCLUDED_TOTAL
        ),
    },
    {
        "check": (
            "development_total"
        ),
        "actual": (
            len(
                development_base
            )
        ),
        "expected": (
            309
        ),
    },
    {
        "check": (
            "external_total"
        ),
        "actual": (
            len(
                external_locked
            )
        ),
        "expected": (
            180
        ),
    },
    {
        "check": (
            "repeat_count"
        ),
        "actual": (
            len(
                repeat_assignment_signatures
            )
        ),
        "expected": (
            N_REPEATS
        ),
    },
    {
        "check": (
            "total_outer_folds"
        ),
        "actual": (
            len(
                validation_sets
            )
        ),
        "expected": (
            EXPECTED_TOTAL_FOLDS
        ),
    },
    {
        "check": (
            "unique_repeat_assignments"
        ),
        "actual": (
            len(
                set(
                    repeat_assignment_signatures.values()
                )
            )
        ),
        "expected": (
            N_REPEATS
        ),
    },
    {
        "check": (
            "unique_validation_sets"
        ),
        "actual": (
            len(
                set(
                    validation_signatures.values()
                )
            )
        ),
        "expected": (
            EXPECTED_TOTAL_FOLDS
        ),
    },
    {
        "check": (
            "long_assignment_rows"
        ),
        "actual": (
            len(
                all_assignments_long
            )
        ),
        "expected": (
            EXPECTED_LONG_TABLE_ROWS
        ),
    },
    {
        "check": (
            "npz_metadata_audit_pass"
        ),
        "actual": (
            int(
                (
                    npz_metadata_audit_df[
                        "status"
                    ]
                    == "PASS"
                ).sum()
            )
            if len(
                npz_metadata_audit_df
            )
            else EXPECTED_INCLUDED_TOTAL
        ),
        "expected": (
            EXPECTED_INCLUDED_TOTAL
        ),
    },
    {
        "check": (
            "frequency_audit_failures"
        ),
        "actual": (
            len(
                frequency_failures
            )
        ),
        "expected": (
            0
        ),
    },
    {
        "check": (
            "configured_excluded_case_present"
        ),
        "actual": (
            PREPROCESSING_EXCLUDED_ID
            in included_ids
        ),
        "expected": (
            False
        ),
    },
]

final_audit_df = pd.DataFrame(
    final_audit_records
)

final_audit_df[
    "status"
] = np.where(
    final_audit_df[
        "actual"
    ].astype(str)
    == final_audit_df[
        "expected"
    ].astype(str),
    "PASS",
    "FAIL",
)

if (
    final_audit_df[
        "status"
    ]
    != "PASS"
).any():
    raise RuntimeError(
        "最终审计存在失败：\n"
        + final_audit_df.to_string(
            index=False
        )
    )


# =============================================================================
# 17. Excel manifest、配置与README
# =============================================================================

master_hash = master_sha256

dataset_index_hash = sha256_file(
    DATASET_INDEX_CSV
)

npz_audit_hash = sha256_file(
    NPZ_AUDIT_CSV
)

preprocessing_config_hash = sha256_file(
    PREPROCESSING_CONFIG_JSON
)

labels_locked_hash = sha256_file(
    labels_locked_path
)

external_locked_hash = sha256_file(
    external_locked_path
)

summary_df = pd.DataFrame([
    [
        "created_at",
        datetime.now().isoformat(
            timespec="seconds"
        ),
    ],
    [
        "status",
        "FINAL_SPLITS_LOCKED",
    ],
    [
        "master_path",
        str(
            MASTER_XLSX
        ),
    ],
    [
        "master_sha256",
        master_hash,
    ],
    [
        "preprocessed_root",
        str(
            PREPROCESSED_ROOT
        ),
    ],
    [
        "dataset_index_sha256",
        dataset_index_hash,
    ],
    [
        "npz_audit_sha256",
        npz_audit_hash,
    ],
    [
        "preprocessing_config_sha256",
        preprocessing_config_hash,
    ],
    [
        "labels_locked_sha256",
        labels_locked_hash,
    ],
    [
        "external_locked_sha256",
        external_locked_hash,
    ],
    [
        "master_total",
        EXPECTED_MASTER_TOTAL,
    ],
    [
        "master_excluded",
        EXPECTED_MASTER_EXCLUDED,
    ],
    [
        "final_included",
        EXPECTED_INCLUDED_TOTAL,
    ],
    [
        "development",
        309,
    ],
    [
        "external",
        180,
    ],
    [
        "number_of_repeats",
        N_REPEATS,
    ],
    [
        "folds_per_repeat",
        N_OUTER_FOLDS,
    ],
    [
        "total_outer_folds",
        EXPECTED_TOTAL_FOLDS,
    ],
    [
        "outer_split_method",
        "StratifiedKFold",
    ],
    [
        "outer_stratification",
        "model_center × severe_mucositis",
    ],
    [
        "inner_split_method",
        "StratifiedShuffleSplit",
    ],
    [
        "inner_validation_fraction",
        INNER_VALIDATION_FRACTION,
    ],
    [
        "external_center",
        "A",
    ],
    [
        "external_locked",
        True,
    ],
    [
        "configured_excluded_case_present",
        False,
    ],
], columns=[
    "metric",
    "value",
])

manifest_xlsx = (
    TEMP_ROOT
    / "fixed_splits_4x5_manifest.xlsx"
)

with pd.ExcelWriter(
    manifest_xlsx,
    engine="openpyxl",
) as writer:
    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False,
    )

    fold_summary_df.to_excel(
        writer,
        sheet_name="Fold_summary",
        index=False,
    )

    outer_stratum_df.to_excel(
        writer,
        sheet_name="Outer_stratum",
        index=False,
    )

    inner_stratum_df.to_excel(
        writer,
        sheet_name="Inner_stratum",
        index=False,
    )

    wide_assignments.to_excel(
        writer,
        sheet_name="Assignments_wide",
        index=False,
    )

    all_assignments_long.to_excel(
        writer,
        sheet_name="Assignments_long",
        index=False,
    )

    frequency_df.to_excel(
        writer,
        sheet_name="Validation_frequency",
        index=False,
    )

    overlap_counts.to_excel(
        writer,
        sheet_name="Val_overlap_counts",
    )

    overlap_jaccard.to_excel(
        writer,
        sheet_name="Val_overlap_jaccard",
    )

    external_locked.to_excel(
        writer,
        sheet_name="External_locked",
        index=False,
    )

    final_audit_df.to_excel(
        writer,
        sheet_name="Final_audit",
        index=False,
    )

    fold_files_manifest_df.to_excel(
        writer,
        sheet_name="Fold_file_hashes",
        index=False,
    )

    if len(
        npz_metadata_audit_df
    ):
        npz_metadata_audit_df.to_excel(
            writer,
            sheet_name="NPZ_metadata_audit",
            index=False,
        )

style_excel(
    manifest_xlsx
)

config_payload = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),
    "status": (
        "FINAL_SPLITS_LOCKED"
    ),
    "master_path": (
        str(
            MASTER_XLSX
        )
    ),
    "master_sha256": (
        master_hash
    ),
    "expected_master_sha256": (
        EXPECTED_MASTER_SHA256
    ),
    "strict_master_sha256": (
        STRICT_MASTER_SHA256
    ),
    "preprocessed_root": (
        str(
            PREPROCESSED_ROOT
        )
    ),
    "npz_dir": (
        str(
            NPZ_DIR
        )
    ),
    "output_root": (
        str(
            OUTPUT_ROOT
        )
    ),
    "n_repeats": (
        N_REPEATS
    ),
    "n_outer_folds": (
        N_OUTER_FOLDS
    ),
    "total_outer_folds": (
        EXPECTED_TOTAL_FOLDS
    ),
    "outer_split_method": (
        "StratifiedKFold"
    ),
    "outer_shuffle": (
        True
    ),
    "outer_split_seeds": (
        OUTER_SPLIT_SEEDS
    ),
    "inner_split_method": (
        "StratifiedShuffleSplit"
    ),
    "inner_validation_fraction": (
        INNER_VALIDATION_FRACTION
    ),
    "inner_split_seed_formula": (
        "50260700 + repeat*10 + fold"
    ),
    "training_seed_formula": (
        "30260700 + repeat*10 + fold"
    ),
    "stratification": [
        "model_center",
        "severe_mucositis",
    ],
    "development_n": (
        309
    ),
    "external_n": (
        180
    ),
    "external_center": (
        "A"
    ),
    "external_locked": (
        True
    ),
    "expected_development_strata": (
        EXPECTED_DEVELOPMENT_STRATA
    ),
    "expected_external_labels": (
        EXPECTED_EXTERNAL_LABEL_COUNTS
    ),
    "repeat_assignment_signatures_sha256": (
        repeat_assignment_signatures
    ),
    "outer_validation_set_signatures_sha256": (
        validation_signatures
    ),
    "labels_locked_sha256": (
        labels_locked_hash
    ),
    "external_locked_sha256": (
        external_locked_hash
    ),
    "configured_excluded_case_present": (
        False
    ),
}

config_json = (
    TEMP_ROOT
    / "fixed_splits_4x5_config.json"
)

write_json(
    config_payload,
    config_json,
)

readme_text = f"""
NPC重度急性口腔黏膜炎3D dose-map项目：4次重复五折固定划分
================================================================
状态：FINAL_SPLITS_LOCKED

队列
----
总Master：540例
排除：51例
最终纳入：489例
Development：309例
External Center A：180例，完全锁定
配置的排除病例：不存在

Development标签及中心
---------------------
B_label0：42
B_label1：39
C_label0：116
C_label1：112

划分
----
Outer：4 repeats × Stratified 5-fold
分层：model_center × severe_mucositis
Outer split seeds：{OUTER_SPLIT_SEEDS}

Inner：
每个outer train内部固定生成一次80%/20%分层划分。
inner seed = 50260700 + repeat×10 + fold

Training seed：
training seed = 30260700 + repeat×10 + fold
同一fold内所有模型必须使用完全相同的training seed。

文件使用
--------
每个fold：
outer_train.csv
outer_validation.csv
inner_train.csv
inner_validation.csv
inner_earlystop.csv
external_validation.csv

inner_earlystop.csv与inner_validation.csv逐字节相同，
用于兼容旧工程训练代码。

方法学规则
----------
1. M0、M1、M2、M3、M4及全部消融模型必须使用完全相同的20个fold。
2. 同一fold内所有模型必须使用fold_config.json记录的相同training_seed。
3. Stage A只能使用inner_train与inner_validation选择epoch。
4. Stage B使用完整outer_train重新训练，并仅在outer_validation评价。
5. 每个Development患者在每个repeat恰好进入outer validation一次，
   因而每例最终获得4次OOF预测。
6. Center A不得用于训练、早停、epoch选择、模型结构选择、
   阈值选择、概率校准或超参数选择。
7. External只能在模型、阈值及校准全部锁定后一次性评价。
8. 本目录不得覆盖；任何改变均建立新版本目录。
9. 治疗后的end_weight、weight_loss变量未写入正式划分文件。
""".strip()

(
    TEMP_ROOT
    / "README_FIXED_SPLITS.txt"
).write_text(
    readme_text,
    encoding="utf-8",
)


# =============================================================================
# 18. 全目录哈希和最终锁定
# =============================================================================

files_to_hash = sorted(
    path
    for path
    in TEMP_ROOT.rglob(
        "*"
    )
    if (
        path.is_file()
        and
        path.name
        not in {
            "file_hashes_sha256.csv",
            "SPLITS_LOCKED.json",
        }
    )
)

file_hash_dataframe = pd.DataFrame([
    {
        "relative_path": (
            str(
                path.relative_to(
                    TEMP_ROOT
                )
            )
        ),
        "size_bytes": (
            path.stat().st_size
        ),
        "sha256": (
            sha256_file(
                path
            )
        ),
    }
    for path
    in files_to_hash
])

file_hash_path = (
    TEMP_ROOT
    / "file_hashes_sha256.csv"
)

dataframe_to_csv(
    file_hash_dataframe,
    file_hash_path,
)

lock_payload = {
    "locked": (
        True
    ),
    "status": (
        "FINAL_SPLITS_LOCKED"
    ),
    "locked_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),
    "final_total": (
        EXPECTED_INCLUDED_TOTAL
    ),
    "development": (
        309
    ),
    "external": (
        180
    ),
    "n_repeats": (
        N_REPEATS
    ),
    "n_folds_per_repeat": (
        N_OUTER_FOLDS
    ),
    "total_outer_folds": (
        EXPECTED_TOTAL_FOLDS
    ),
    "outer_split_seeds": (
        OUTER_SPLIT_SEEDS
    ),
    "master_sha256": (
        master_hash
    ),
    "labels_locked_sha256": (
        labels_locked_hash
    ),
    "external_locked_sha256": (
        external_locked_hash
    ),
    "manifest_sha256": (
        sha256_file(
            manifest_xlsx
        )
    ),
    "config_sha256": (
        sha256_file(
            config_json
        )
    ),
    "all_assignments_sha256": (
        sha256_file(
            all_assignments_long_path
        )
    ),
    "file_hashes_manifest_sha256": (
        sha256_file(
            file_hash_path
        )
    ),
    "external_center": (
        "A"
    ),
    "external_locked": (
        True
    ),
    "configured_excluded_case_present": (
        False
    ),
}

lock_json = (
    TEMP_ROOT
    / "SPLITS_LOCKED.json"
)

write_json(
    lock_payload,
    lock_json,
)

run_log_lines = [
    (
        "5-fold × 4 repeats split generation"
    ),
    (
        "=" * 90
    ),
    (
        "Created at: "
        + datetime.now().isoformat(
            timespec="seconds"
        )
    ),
    (
        f"Python: {sys.version.split()[0]}"
    ),
    (
        f"Master: {MASTER_XLSX}"
    ),
    (
        f"Master SHA256: {master_hash}"
    ),
    (
        f"Preprocessed root: {PREPROCESSED_ROOT}"
    ),
    (
        f"Output root: {OUTPUT_ROOT}"
    ),
    (
        "Total/Excluded/Included: 540/51/489"
    ),
    (
        "Development/External: 309/180"
    ),
    (
        "Development strata: "
        f"{EXPECTED_DEVELOPMENT_STRATA}"
    ),
    (
        "External labels: "
        f"{EXPECTED_EXTERNAL_LABEL_COUNTS}"
    ),
    (
        "Outer CV: 4 repeats × 5 folds"
    ),
    (
        "Outer split seeds: "
        f"{OUTER_SPLIT_SEEDS}"
    ),
    (
        "Inner validation fraction: "
        f"{INNER_VALIDATION_FRACTION}"
    ),
    (
        "NPZ metadata audit: "
        f"{EXPECTED_INCLUDED_TOTAL}/"
        f"{EXPECTED_INCLUDED_TOTAL} PASS"
    ),
    (
        "Each Development patient outer "
        "validation frequency: 4"
    ),
    (
        "Each Development patient outer "
        "train frequency: 16"
    ),
    (
        "External center A locked: True"
    ),
    (
        "Configured excluded case present: False"
    ),
    (
        "Status: FINAL_SPLITS_LOCKED"
    ),
]

run_log_path = (
    TEMP_ROOT
    / "fixed_splits_4x5_run_log.txt"
)

run_log_path.write_text(
    "\n".join(
        run_log_lines
    ),
    encoding="utf-8",
)


# =============================================================================
# 19. 原子完成：临时目录重命名为正式目录
# =============================================================================

TEMP_ROOT.rename(
    OUTPUT_ROOT
)

print()
print("=" * 96)
print(
    "4次重复五折固定划分生成并冻结完成"
)
print("=" * 96)
print(
    "Development：309例"
)
print(
    "External：180例，Center A完全锁定"
)
print(
    "Outer CV：4 repeats × 5 folds = 20 folds"
)
print(
    "每个Development病例进入outer validation：4次"
)
print(
    "每个Development病例进入outer train：16次"
)
print(
    "最终分析NPZ内部复核：489/489通过（预处理池保留497例）"
    if AUDIT_NPZ_METADATA
    else "NPZ内部复核：未运行"
)
print(
    "20个outer validation集合：20/20均不同"
)
print(
    "4个repeat完整fold assignment：4/4均不同"
)
print(
    "配置的排除病例：不存在"
)
print(
    "最终审计：全部PASS"
)
print()
print(
    "输出目录：",
    OUTPUT_ROOT,
)
print(
    "核心报告：",
    OUTPUT_ROOT
    / "fixed_splits_4x5_manifest.xlsx",
)
print(
    "锁定文件：",
    OUTPUT_ROOT
    / "SPLITS_LOCKED.json",
)
print()
print(
    "repeat_01/fold_01训练文件："
)
print(
    OUTPUT_ROOT
    / "repeat_01"
    / "fold_01"
    / "outer_train.csv"
)
print(
    OUTPUT_ROOT
    / "repeat_01"
    / "fold_01"
    / "outer_validation.csv"
)
print(
    OUTPUT_ROOT
    / "repeat_01"
    / "fold_01"
    / "inner_train.csv"
)
print(
    OUTPUT_ROOT
    / "repeat_01"
    / "fold_01"
    / "inner_validation.csv"
)